# Failed EDAs

EDA attempts that didn't work, kept as evidence for the decisions they led to.

**Not runnable on its own.** The cells use `DATA`, `muse` and `seed_counts`, which are defined in the Setup and MuSe sections of `DAP_IEDA_Template.ipynb`.

## Last.fm tags: sample fetch (1,000 songs)

> **Sample only.** This checks that we *can* fetch each song's tags from the Last.fm API, and that the tags are rich enough to weight and analyse. It is not the full catalogue fetch.

MuSe keeps only the mood word (seed) that found each song, not the song's full tag list. So for a random sample of **1,000 MuSe songs** we call [`track.getTopTags`](https://www.last.fm/api/show/track.getTopTags) with the song's `artist` and `track` (`autocorrect=1` fixes small spelling differences).

Each tag comes with a `count`: a **relative weight from 0 to 100**, where the song's most-used tag is 100.

**Setup:** paste your key into `mood-playlist/.env` as `LASTFM_API_KEY=...` (the file is git-ignored). Every response is cached to `data/raw/lastfm/sample_1000.jsonl`, so re-running the notebook makes no API calls. An interrupted fetch picks up where it stopped.

In [ ]:
import json
import time
import urllib.parse
import urllib.request
from urllib.error import HTTPError, URLError

INTERIM = Path("../data/interim")
LASTFM_CACHE = DATA / "lastfm" / "sample_1000.jsonl"
N_SAMPLE = 1000
MIN_COUNT = 5  # a tag "counts" if its weight is at least 5 (of 100)
MIN_TAGS = 5   # a song is usable if it has at least 5 such tags, seed words excluded

sample = muse.sample(N_SAMPLE, random_state=42)[["lastfm_url", "artist", "track", "seeds"]]


def read_env(path):
    """Minimal .env reader: KEY=value lines, # comments."""
    env = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            env[key.strip()] = value.strip().strip("\"'")
    return env


def get_top_tags(artist, track, api_key, retries=5):
    """One track.getTopTags call. Returns the parsed JSON (which may hold a Last.fm error)."""
    query = urllib.parse.urlencode({"method": "track.gettoptags", "artist": artist, "track": track,
                                    "autocorrect": 1, "api_key": api_key, "format": "json"})
    request = urllib.request.Request(f"https://ws.audioscrobbler.com/2.0/?{query}",
                                     headers={"User-Agent": "MoodMix-student-project/0.1"})
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(request, timeout=20) as r:
                data = json.load(r)
        except HTTPError as e:  # Last.fm sends errors as JSON with a 4xx/5xx status
            try:
                data = json.load(e)
            except ValueError:
                data = {"error": e.code, "message": str(e)}
        except (URLError, TimeoutError, ValueError) as e:
            data = {"error": -1, "message": str(e)}
        # 29 = rate limited, 11/16 = service temporarily down, -1/5xx = network: wait and retry
        if data.get("error") in (29, 11, 16, -1) or data.get("error", 0) >= 500:
            time.sleep(2 ** attempt)
            continue
        return data
    return data

In [ ]:
# Fetch every sampled song not already in the cache (~4 requests/second, ~5 minutes for 1,000).
LASTFM_CACHE.parent.mkdir(parents=True, exist_ok=True)
done = set()
if LASTFM_CACHE.exists():
    with LASTFM_CACHE.open(encoding="utf-8") as f:
        done = {json.loads(line)["lastfm_url"] for line in f}

todo = sample[~sample["lastfm_url"].isin(done)]
if len(todo):
    api_key = read_env("../.env").get("LASTFM_API_KEY")
    assert api_key, "Paste your key into mood-playlist/.env as LASTFM_API_KEY=..."
    with LASTFM_CACHE.open("a", encoding="utf-8") as f:
        for i, row in enumerate(todo.itertuples(), 1):
            started = time.monotonic()
            response = get_top_tags(row.artist, row.track, api_key)
            f.write(json.dumps({"lastfm_url": row.lastfm_url, "response": response}) + "\n")
            f.flush()
            if i % 100 == 0:
                print(f"{i}/{len(todo)} fetched")
            time.sleep(max(0, 0.25 - (time.monotonic() - started)))
print(f"cache: {len(done) + len(todo):,} songs ({len(todo):,} fetched now)")

#### Tidy tables

One row per (song, tag) pair in `lastfm_tags_sample.csv`, and one status per song in `lastfm_status_sample.csv`:
- `ok`: the song was found and has at least one tag
- `no_tags`: the song was found but nobody has tagged it
- `not_found`: Last.fm doesn't know the song (error 6)
- `error`: any other API error, after retries

Tag names are lower-cased and trimmed, so *Sad* and *sad* are one tag (we keep the higher weight).

In [ ]:
tag_rows, status_rows = [], []
with LASTFM_CACHE.open(encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        url, resp = rec["lastfm_url"], rec["response"]
        if "error" in resp:
            status = "not_found" if resp["error"] == 6 else "error"
        else:
            tags = resp.get("toptags", {}).get("tag", [])
            tags = [tags] if isinstance(tags, dict) else tags  # a single tag comes back as an object
            tag_rows += [(url, t["name"].strip().lower(), int(t["count"])) for t in tags]
            status = "ok" if tags else "no_tags"
        status_rows.append((url, status))

lfm_tags = (pd.DataFrame(tag_rows, columns=["lastfm_url", "tag", "count"])
            .sort_values("count", ascending=False).drop_duplicates(["lastfm_url", "tag"]))
lfm_status = pd.DataFrame(status_rows, columns=["lastfm_url", "status"]).drop_duplicates("lastfm_url", keep="last")
lfm_tags.to_csv(INTERIM / "lastfm_tags_sample.csv", index=False)
lfm_status.to_csv(INTERIM / "lastfm_status_sample.csv", index=False)
lfm_tags.head()

#### Are the tags usable?

A song is **usable** if, after removing its own seed word(s), it still has at least `MIN_TAGS` = 5 tags with weight `MIN_COUNT` ≥ 5. That is the input the Song Encoder would see.

We also check how often the seed itself appears in the song's tags. A high rate confirms the leakage risk in the proposal: if the seed stays in the input, the model can just copy it.

In [ ]:
seed_vocab = set(seed_counts.index)
lfm = lfm_tags.merge(sample[["lastfm_url", "seeds"]], on="lastfm_url")
lfm["is_seed"] = [t in s for t, s in zip(lfm["tag"], lfm["seeds"])]
strong = lfm[lfm["count"] >= MIN_COUNT]

per_song = sample[["lastfm_url"]].merge(lfm_status, on="lastfm_url", how="left").fillna({"status": "not fetched"})
per_song["tags_all"] = per_song["lastfm_url"].map(lfm.groupby("lastfm_url").size()).fillna(0).astype(int)
per_song["tags_strong_no_seed"] = (per_song["lastfm_url"]
                                   .map(strong[~strong["is_seed"]].groupby("lastfm_url").size()).fillna(0).astype(int))
per_song["seed_in_tags"] = per_song["lastfm_url"].isin(lfm.loc[lfm["is_seed"], "lastfm_url"])
per_song["usable"] = per_song["tags_strong_no_seed"] >= MIN_TAGS

display(per_song["status"].value_counts().to_frame("songs").assign(
    pct=lambda d: (d["songs"] / len(per_song) * 100).round(1)).T)
ok = per_song[per_song["status"] == "ok"]
pd.Series({
    "usable songs (of sample)": f"{per_song['usable'].sum():,} ({per_song['usable'].mean():.1%})",
    "seed word among the song's tags (of found songs)": f"{ok['seed_in_tags'].mean():.1%}",
    "median tags per found song (all weights)": ok["tags_all"].median(),
    f"median tags per found song (weight >= {MIN_COUNT}, seed removed)": ok["tags_strong_no_seed"].median(),
    "distinct tags in sample": f"{lfm['tag'].nunique():,}",
    "share of tags (weight >= 5) that are also MuSe seed words": f"{strong['tag'].isin(seed_vocab).mean():.1%}",
}).to_frame("value")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1, 1, 1.2]})

axes[0].hist(ok["tags_strong_no_seed"].clip(upper=60), bins=np.arange(-0.5, 61.5, 2), color=BLUE)
axes[0].axvline(MIN_TAGS, color=ORANGE, lw=1.5, ls="--", label=f"usable cut-off ({MIN_TAGS})")
axes[0].set_title(f"Tags per found song (weight >= {MIN_COUNT}, seed removed)")
axes[0].set_xlabel("tags (clipped at 60)")
axes[0].set_ylabel("songs")
axes[0].legend()
axes[0].grid(axis="x", visible=False)

axes[1].hist(lfm["count"], bins=np.arange(-0.5, 101.5, 5), color=BLUE)
axes[1].set_title("Tag weights")
axes[1].set_xlabel("count (0-100, top tag = 100)")
axes[1].set_ylabel("tags")
axes[1].set_yscale("log")
axes[1].grid(axis="x", visible=False)

top = strong.groupby("tag")["lastfm_url"].nunique().nlargest(30).sort_values()
axes[2].barh(top.index, top.values, color=[BLUE if t in seed_vocab else GREY for t in top.index])
axes[2].set_title(f"Top 30 tags (songs with weight >= {MIN_COUNT})")
axes[2].set_xlabel("songs")
axes[2].tick_params(axis="y", labelsize=8)
axes[2].legend(handles=[Patch(color=BLUE, label="MuSe mood word"), Patch(color=GREY, label="other")], loc="lower right")
axes[2].grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

**Response.**

**The fetch failed: Last.fm no longer returns track tags for almost any song.**

- **Fetching works?** The API itself works: all 506 songs fetched were found, with no API errors and normal listener counts. But **500 of 506 (98.8%) came back with an empty tag list.**
- **Which songs still have tags?** Only 6, all big hits: Radiohead "Fake Plastic Trees", Pitbull "Time of Our Lives", Elliott Smith "Everything Reminds Me of Her", Songs: Ohia "Farewell Transmission", Halsey "Without Me" and Taylor Swift "Love Story". Each got exactly 10 tags.
- **Sample size.** The cache holds 506 of the planned 1,000 songs. With 98.8% empty, fetching the rest wouldn't change the conclusion.
- **Usable tags, leakage check.** Can't be measured on 6 songs.
- **Decision.** The Song Encoder reads **lyrics** instead of tags. See the lyrics section in `DAP_IEDA_Template.ipynb` and the v2 plan in `project-idea.md`.